# Челленджи недели: память, потоки, процессы и asyncio

**Цель:** проверить, что концепты concurrency работают вместе и применяются к реальным задачам — гонкам данных, параллельной обработке, асинхронному I/O.

**Как работать:**
- Часть задач сопровождается ячейкой «Объясни своими словами» — там нужно не только написать код, но и сформулировать, почему он работает.
- Задачи решаются на 5-30 строк кода. На W4 разрешены все Python-фичи: `def`, `class`, `async def`, `threading`, `multiprocessing`, `asyncio`.
- Сначала попробуй сам, без подглядываний. Если застрял на 10+ минут — открой solution-версию.

**Где решать.** Большинство заданий — прямо в ячейках ноутбука. Исключение — задание 2 (`multiprocessing.Pool`). Его нужно решать в отдельном `.py`-файле, в задании увидишь блок:

> **→ Перейди в терминал.** `python scripts/task_02_pool_practice.py`

Причина — `Pool` в Jupyter на Windows не работает (дочерние процессы не видят функции, объявленные в ячейках ноутбука). Подробности — в `scripts/README.md`.

**Setup для Jupyter.** Asyncio-задачи зовут `asyncio.run(...)` внутри ячеек. Jupyter уже крутит свой event loop, поэтому нужен `nest_asyncio.apply()` ниже. Если пакет не установлен — `pip install nest_asyncio`.


In [ ]:
# Технический setup для Jupyter — это НЕ материал недели, а среда исполнения.
# Одна строка нужна, чтобы ноутбук вообще запустился; ничего здесь учить не надо.
#
# nest_asyncio.apply() — Jupyter уже крутит свой event loop в фоне, и
# `asyncio.run(...)` внутри ячейки иначе падает с RuntimeError. Этот патч
# разрешает повторный запуск loop'а поверх существующего. В обычном `.py`
# скрипте такая строка не нужна.
import nest_asyncio
nest_asyncio.apply()
print("setup OK")


## Задание 1: Потокобезопасный счётчик через `Lock`

Реализуй класс `SafeCounter` с методом `increment()`, который безопасно увеличивает внутренний счётчик из нескольких потоков. Внутри используй `threading.Lock` через `with self._lock:`.

Затем запусти 10 потоков, каждый из которых делает 10 000 инкрементов. Финальное значение должно быть ровно `100_000` — это и есть проверка, что race condition нет.

Подсказка: без `Lock` некоторые инкременты потеряются (`counter += 1` это три шага: read / add / write).

In [ ]:
import threading

# TODO: implement
# your code:


**Объясни своими словами:** что произойдёт, если убрать `with self._lock:` из метода `increment`?

*Опиши здесь своими словами:*

## Задание 2: CPU-bound — параллельная сумма квадратов через `Pool`

Реализуй функцию `heavy_sum_of_squares(n)`, которая считает `sum(i*i for i in range(n))`. На больших `n` (несколько миллионов) это чисто CPU-задача.

Затем сравни два способа:

1. **Последовательно**: прогон функции на 4 значениях `n` через обычный `for`.
2. **Параллельно через `multiprocessing.Pool(4)`**: те же 4 значения через `pool.map`.

Замерь время через `time.perf_counter()` и выведи speedup (`seq_time / pool_time`). На 4-ядерной машине ожидаем ~3-4x ускорения.

> **→ Перейди в терминал.** Задание решается в отдельном `.py`-файле — шаблон с `TODO` уже лежит рядом. Открой `scripts/task_02_pool_practice.py` в редакторе, заполни блоки `TODO` и запусти из папки `notebooks/`:
>
> ```bash
> python scripts/task_02_pool_practice.py
> ```
>
> Когда увидишь speedup и `результаты совпали: True` — сравни своё решение с `scripts/task_02_pool_solution.py` и возвращайся к следующему заданию ноутбука.

**Почему отдельный скрипт.** В Jupyter `multiprocessing.Pool` кросс-платформенно не работает: на Windows дочерние процессы не видят функции, объявленные в ячейках ноутбука. Поэтому правильный паттерн для `Pool` — `.py`-файл с гвардом `if __name__ == "__main__":`.

**Объясни своими словами:** почему здесь нужен `multiprocessing.Pool`, а не `ThreadPoolExecutor`?

*Опиши здесь своими словами:*

## Задание 3: Параллельные HTTP-запросы через `asyncio.gather`

Симулируй параллельные HTTP-запросы. Без реальной сети — используем `asyncio.sleep(latency)` как заглушку.

- Напиши `async def fetch(url, latency)` — печатает «start url», засыпает на `latency` секунд через `await asyncio.sleep(latency)`, печатает «done url» и возвращает строку `f"<{url}>"`.
- Напиши `async def main()` — берёт 4 URL'а с разными latency (0.3, 0.2, 0.4, 0.1 с), запускает `asyncio.gather(...)` параллельно, замеряет время, печатает результаты и общее время.
- Запусти через `asyncio.run(main())`. Общее время должно быть ≈0.4с (равно самой медленной), а не 1.0с (сумма).

In [ ]:
import asyncio
import time

# TODO: implement
# your code:


## Задание 4: Rate-limiting через `asyncio.Semaphore`

Бывает: API позволяет максимум 3 одновременных запроса; больше — даёт `429 Too Many Requests`. Решение — `asyncio.Semaphore(3)`: семафор пропускает не больше 3 параллельно, остальные ждут освобождения слота.

- Напиши `async def fetch(sem, idx)` — внутри `async with sem:` печатает «start idx», засыпает на `0.2` с, печатает «done idx».
- В `main()` запусти 10 задач через `asyncio.gather`. Семафор должен пропускать только 3 одновременно — посмотри по принту, как они идут волнами по 3.

Замерь общее время — для 10 задач × 0.2с при лимите 3 ожидаем `~ceil(10/3) * 0.2 = 0.8с`.

In [ ]:
import asyncio
import time

# TODO: implement
# your code:


## Задание 5: Блокирующий код через `asyncio.to_thread`

Бывает: нужно из `async def` позвать обычную блокирующую функцию (нет async-версии библиотеки или legacy-код). Если позвать напрямую — она заморозит event loop на всё время выполнения. Решение — `asyncio.to_thread(func, *args)`: выносит вызов в отдельный поток, event loop остаётся свободным.

- Напиши блокирующую `def blocking_compute(name, seconds)` — `time.sleep(seconds)` + возвращает строку.
- В `async def main()` запусти 3 таких вызова **параллельно** через `asyncio.gather(*(asyncio.to_thread(blocking_compute, name, sec) for ...))`.
- Замерь время — несмотря на то что `blocking_compute` блокирующая, через `to_thread` три вызова идут параллельно (общее время ~max).

In [ ]:
import asyncio
import time

# TODO: implement
# your code:


## Задание 6: Асинхронный генератор

Асинхронный генератор — это функция, объявленная через `async def`, которая использует `yield`. По ней итерируются через `async for`.

- Реализуй `async def stream_events(n)` — асинхронный генератор, который выдаёт `n` событий: между ними `await asyncio.sleep(0.05)` (имитация прихода события из сети).
- Каждое событие — словарь `{"id": i, "timestamp": time.perf_counter()}`.
- В `main()` итерируй через `async for event in stream_events(5):` и печатай каждое событие.

Подсказка: внутри `async def` с `yield` нельзя писать `return value` — это синтаксическая ошибка для async-генераторов. Только `yield`.

In [ ]:
import asyncio
import time

# TODO: implement
# your code:


## Задание 7: Циклическая ссылка и сборщик мусора

Создай два объекта, ссылающихся друг на друга — циклическая ссылка. Покажи, что после `del` обычных переменных счётчик ссылок не падает до 0 (объекты живы), и только `gc.collect()` их освобождает.

- Сделай простой класс `Node` с атрибутом `partner` и `__del__`-методом, который печатает «удалили <name>» (так увидим момент освобождения).
- Создай `a = Node("A")`, `b = Node("B")`, свяжи их через `a.partner = b; b.partner = a`.
- Удали локальные ссылки: `del a; del b`. До `gc.collect()` `__del__` не вызывался — это видно по отсутствию принта.
- Вызови `gc.collect()` — оба `__del__` напечатают своё сообщение.

In [ ]:
import gc

# TODO: implement
# your code:


**Объясни своими словами:** почему `del a; del b` сами по себе не освободили объекты, а `gc.collect()` смог?

*Опиши здесь своими словами:*

# Готово

Ты только что прошёл задачи на пересечении модели памяти, потоков, процессов и asyncio. Race condition с `Lock`, параллельная обработка CPU-задач через `Pool`, асинхронные HTTP-вызовы через `gather`, rate-limiting через `Semaphore`, блокирующий код через `to_thread`, async-генераторы и циклические ссылки — это набор паттернов, которые встречаются в backend и ML-инфраструктуре каждый день.

На следующей неделе мы переключимся на специфику вашего трека — дообучение языковых моделей (NLP fine-tuning). Concurrency-инструменты из этой недели уйдут в фон — они там везде, но больше не будут главной темой.
